# Tutorial 6: Memory Systems in LangChain

In this tutorial, we'll explore memory systems in LangChain, which allow us to create more contextual and personalized interactions with language models.

## 1. Types of Memory in LangChain + LangGraph

In modern LangChain 1.0 + LangGraph 1.0, memory is handled differently from older versions:

| Type | Implementation | Scope |
|---|---|---|
| **Short-term (buffer)** | `MemorySaver` checkpointer + `Annotated[List, add_messages]` | Per thread |
| **Summary memory** | LCEL chain that summarises history before each call | Per thread |
| **Long-term** | `InMemoryStore` or `SqliteSaver` | Cross-thread |
| **Context trimming** | `trim_messages()` | Per call |

The old `ConversationBufferMemory`, `ConversationSummaryMemory`, and `ConversationChain` classes have been removed. Use LangGraph state + checkpointers instead.

In [1]:
import os
import operator
from typing import List, Annotated, TypedDict
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage, trim_messages
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.1)
print("Setup complete.")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Setup complete.


## 2. Buffer Memory with LangGraph

In LangGraph, short-term (buffer) memory is built into the state using `Annotated[List, operator.add]` for message accumulation, combined with a `MemorySaver` checkpointer.

In [2]:
class ConversationState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]

def chat_node(state: ConversationState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [AIMessage(content=response.content)]}

chat_workflow = StateGraph(ConversationState)
chat_workflow.add_node("chat", chat_node)
chat_workflow.set_entry_point("chat")
chat_workflow.add_edge("chat", END)

# MemorySaver persists the full message history across turns
chat_app = chat_workflow.compile(checkpointer=MemorySaver())

config = {"configurable": {"thread_id": "alice-session"}}

r1 = chat_app.invoke({"messages": [HumanMessage(content="Hi, my name is Alice.")]}, config)
print("Turn 1:", r1["messages"][-1].content[:150])

r2 = chat_app.invoke({"messages": [HumanMessage(content="What is my name?")]}, config)
print("Turn 2:", r2["messages"][-1].content[:150])

Turn 1: Hello Alice, it's nice to meet you. Is there something I can help you with or would you like to chat?
Turn 2: Your name is Alice.


## 3. Summary Memory

For long conversations, summarising older messages keeps the context window small. We run a summarisation step before each call when the history exceeds a threshold.

In [3]:
class SummaryState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]
    summary: str

MAX_MESSAGES = 6  # summarise when history exceeds this

def summarise_history(messages: list, existing_summary: str) -> str:
    prompt = f"""
Summarise the conversation so far (existing summary may be empty).
Existing summary: {existing_summary}
New messages: {chr(10).join(m.content for m in messages)}
Write a concise 2-3 sentence summary."""
    return llm.invoke([HumanMessage(content=prompt)]).content

def summary_chat_node(state: SummaryState) -> dict:
    messages = state["messages"]
    summary = state.get("summary", "")

    # If history is long, summarise older messages
    if len(messages) > MAX_MESSAGES:
        older = messages[:-2]
        recent = messages[-2:]
        summary = summarise_history(older, summary)
        messages = recent

    # Prepend summary as system message if available
    context = ([SystemMessage(content=f"Conversation so far: {summary}")] if summary else []) + messages
    response = llm.invoke(context)
    return {"messages": [AIMessage(content=response.content)], "summary": summary}

summary_workflow = StateGraph(SummaryState)
summary_workflow.add_node("chat", summary_chat_node)
summary_workflow.set_entry_point("chat")
summary_workflow.add_edge("chat", END)
summary_app = summary_workflow.compile(checkpointer=MemorySaver())

config2 = {"configurable": {"thread_id": "summary-session"}}
r = summary_app.invoke({"messages": [HumanMessage(content="Tell me about Python.")], "summary": ""}, config2)
print("Summary chat:", r["messages"][-1].content[:200])

Summary chat: **Python Overview**

Python is a high-level, interpreted programming language that is widely used for various purposes such as web development, scientific computing, data analysi


## 4. Long-term Memory with InMemoryStore

`InMemoryStore` provides cross-thread storage — facts saved in one session are available in all future sessions for the same user.

In [4]:
import uuid

lt_store = InMemoryStore()  # accessible via closure

class LongTermState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]
    user_id: str

def long_term_chat(state: LongTermState) -> dict:
    user_id = state["user_id"]
    namespace = ("users", user_id, "facts")

    # Load stored facts
    facts = [m.value.get("fact", "") for m in lt_store.search(namespace)]
    system = f"Known facts: {'; '.join(facts)}" if facts else "No known facts yet."

    full = [SystemMessage(content=system)] + state["messages"]
    response = llm.invoke(full)

    # Save new user facts
    last = state["messages"][-1].content
    if any(kw in last.lower() for kw in ["my name", "i like", "i love", "i am", "i prefer"]):
        lt_store.put(namespace, str(uuid.uuid4()), {"fact": last})

    return {"messages": [AIMessage(content=response.content)]}

lt_workflow = StateGraph(LongTermState)
lt_workflow.add_node("chat", long_term_chat)
lt_workflow.set_entry_point("chat")
lt_workflow.add_edge("chat", END)
lt_app = lt_workflow.compile(checkpointer=MemorySaver())

uid = "user-bob"
# Session 1: Bob introduces himself
r1 = lt_app.invoke({"messages": [HumanMessage(content="I am Bob and I love hiking.")], "user_id": uid},
                   {"configurable": {"thread_id": "bob-lt-s1"}})
print("Session 1:", r1["messages"][-1].content[:150])

# Session 2: Different thread, same user — facts are remembered via lt_store
r2 = lt_app.invoke({"messages": [HumanMessage(content="What do you know about me?")], "user_id": uid},
                   {"configurable": {"thread_id": "bob-lt-s2"}})
print("Session 2 (new thread):", r2["messages"][-1].content[:200])

Session 1: Nice to meet you, Bob. It sounds like you enjoy spending time outdoors and exploring nature through hiking. What's your favorite type of hike? Do you 
Session 2 (new thread): You've told me that your name is Bob, and you love hiking. That's the extent of my knowledge about you so far. Would you like to share more about yourself?


In [5]:
# Show what was stored in long-term memory
stored = lt_store.search(("users", uid, "facts"))
print(f"Facts stored for {uid}:")
for item in stored:
    print(f"  - {item.value['fact']}")

Facts stored for user-bob:
  - I am Bob and I love hiking.


## Conclusion

In this tutorial, we've explored various memory systems in LangChain, from simple buffer memory to more advanced techniques like combined memory with custom prompts. These memory systems allow you to create more contextual and personalized interactions in your language model applications.